# Find Homography So Dope — Complete Submission Notebook

**Objective**: Predict the $3 \times 3$ homography matrix $H$ (with $h_{33} = 1$) mapping pixel coordinates from `image_1` → `image_2` for each given image pair.

$$
\begin{bmatrix} x_2' \\ y_2' \\ w_2' \end{bmatrix} = 
\begin{bmatrix} h_{11} & h_{12} & h_{13} \\ h_{21} & h_{22} & h_{23} \\ h_{31} & h_{32} & 1 \end{bmatrix}
\begin{bmatrix} x_1 \\ y_1 \\ 1 \end{bmatrix}, \quad x_2 = \frac{x_2'}{w_2'}, \quad y_2 = \frac{y_2'}{w_2'}
$$

### End-to-End Pipeline
1. **Environment & Path Auto-Discovery**: Seamless execution on Kaggle (auto-detects dataset paths) or local machines.
2. **Evaluation Metric**: Geometric reprojection error across 5 canonical normalized coordinates.
3. **Exploratory Data Analysis**: Sequence structure, illumination vs. viewpoint analysis, identity prior.
4. **Multi-Stage Feature Matching**:
   - Stage 1: Standard SIFT (5000 features, Lowe ratio 0.75)
   - Stage 2: CLAHE contrast boost + RootSIFT (Hellinger kernel normalization)
   - Stage 3: Relaxed matching for challenging textures
   - Stage 4: Aggressive CLAHE + sensitive SIFT
   - Fallback: Identity matrix prior
5. **Geometric Guardrails**: Positive projective depth ($w > 0$), convex quadrilateral check, determinant range $(0.005, 50.0)$, bounding box sanity.
6. **Local Validation**: Full benchmark on training set (140 pairs).
7. **Visualizations**: Aligned warped overlays and SIFT keypoint matches.
8. **Test Inference & Submission**: Generates verified `submission.csv` ready for Kaggle leaderboard upload.

---
## 0. Setup & Imports

In [ ]:
import os
import sys
import glob
import time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Visualization defaults
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['image.interpolation'] = 'nearest'

print(f"Python  : {sys.version.split()[0]}")
print(f"OpenCV  : {cv2.__version__}")
print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")
print(f"MAGSAC  : {'Available' if hasattr(cv2, 'USAC_MAGSAC') else 'Fallback to RANSAC'}")

---
## 1. Environment & Path Auto-Discovery

This notebook automatically detects whether it is running on **Kaggle** or **locally**, discovers where the competition data is stored, and verifies image paths.

In [ ]:
# ==============================================================================
# ROBUST PATH AUTO-DISCOVERY & ENVIRONMENT SETUP
# ==============================================================================

IS_KAGGLE = os.path.exists('/kaggle')
print(f"Detected Platform: {'Kaggle Notebook' if IS_KAGGLE else 'Local Environment'}")

KAGGLE_COMPETITION_ROOT = '/kaggle/input/find-homography-so-dope'
SEARCH_ROOTS = [
    KAGGLE_COMPETITION_ROOT,
    '/kaggle/input',
    '/kaggle/working',
    '.',
    '..',
    os.path.abspath(os.path.join(os.getcwd(), '..')),
]

# --- Step 1: Find test.csv ---
test_csv_candidates = []
for root in SEARCH_ROOTS:
    if os.path.exists(root):
        direct = os.path.join(root, 'test.csv')
        if os.path.exists(direct) and direct not in test_csv_candidates:
            test_csv_candidates.append(direct)
        for p in glob.glob(os.path.join(root, '**', 'test.csv'), recursive=True):
            if p not in test_csv_candidates:
                test_csv_candidates.append(p)

TEST_CSV = test_csv_candidates[0] if test_csv_candidates else None

if TEST_CSV is None:
    print("=" * 76)
    print("⚠️  DATASET NOT FOUND IN KAGGLE / LOCAL WORKSPACE")
    print("=" * 76)
    if IS_KAGGLE:
        print("\nYou are running in a Kaggle notebook, but the competition data")
        print("has not been added as an Input to this notebook yet.\n")
        print("👉 HOW TO FIX (Takes 15 seconds):")
        print("   1. Look at the right-hand panel of this Kaggle notebook.")
        print("   2. In the 'Input' section, click '+ Add Input' (or '+ Add Data').")
        print("   3. In the popup dialog, select the 'Competition' tab.")
        print("   4. Type 'find-homography-so-dope' in the search bar.")
        print("   5. Click the '+' (Add) button next to the competition name.")
        print("   6. Once added, re-run this cell!\n")
        print("Current contents of /kaggle/input/:")
        if os.path.exists('/kaggle/input'):
            items = os.listdir('/kaggle/input')
            if items:
                for item in items:
                    p = os.path.join('/kaggle/input', item)
                    contents = os.listdir(p)[:5] if os.path.isdir(p) else []
                    print(f"   - /kaggle/input/{item} (files: {contents})")
            else:
                print("   (Empty folder — no datasets currently attached)")
        print("=" * 76)
    else:
        print(f"\nLocal directory: {os.getcwd()}")
        print("Please ensure this notebook is executed from the repository root")
        print("containing 'test.csv' and 'data/'.")
        print("=" * 76)
    raise FileNotFoundError("Competition data not found. Please follow the instructions above.")

BASE_DIR = os.path.dirname(os.path.abspath(TEST_CSV))
test_df = pd.read_csv(TEST_CSV)
print(f"[OK] Found test.csv: {TEST_CSV} ({len(test_df)} test pairs)")

# --- Step 2: Find train.csv (optional for submission, used for validation) ---
train_candidates = [
    os.path.join(BASE_DIR, 'train.csv'),
    os.path.join(BASE_DIR, 'data', 'train.csv'),
]
for root in SEARCH_ROOTS:
    if os.path.exists(root):
        direct = os.path.join(root, 'train.csv')
        if os.path.exists(direct) and direct not in train_candidates:
            train_candidates.append(direct)
        for p in glob.glob(os.path.join(root, '**', 'train.csv'), recursive=True):
            if p not in train_candidates:
                train_candidates.append(p)

TRAIN_CSV = None
for c in train_candidates:
    if os.path.exists(c):
        TRAIN_CSV = c
        break

train_df = None
if TRAIN_CSV and os.path.exists(TRAIN_CSV):
    train_df = pd.read_csv(TRAIN_CSV)
    print(f"[OK] Found train.csv: {TRAIN_CSV} ({len(train_df)} train pairs)")
else:
    print("[INFO] train.csv not found. (Local validation will be skipped; test inference is ready).")

# --- Step 3: Automatically resolve image directories on disk ---
def resolve_image_root(sample_rel_path, candidate_roots):
    norm_sample = os.path.normpath(sample_rel_path)
    for b in candidate_roots:
        if not b or not os.path.exists(b):
            continue
        if os.path.exists(os.path.join(b, norm_sample)):
            return b
        for sub in ['data/test', 'data/train', 'data', 'test', 'train', 'test_images', 'train_images']:
            cand = os.path.join(b, sub)
            if os.path.exists(os.path.join(cand, norm_sample)):
                return cand
    for root in SEARCH_ROOTS:
        if os.path.exists(root):
            matches = glob.glob(os.path.join(root, '**', os.path.basename(norm_sample)), recursive=True)
            for m in matches:
                target_sub = norm_sample.replace('\\', '/')
                m_sub = m.replace('\\', '/')
                if target_sub in m_sub:
                    idx = m_sub.rfind(target_sub)
                    return m[:idx].rstrip('/\\')
    return None

sample_test_img = test_df.iloc[0]['image_1']
TEST_IMG_DIR = resolve_image_root(sample_test_img, [
    os.path.join(BASE_DIR, 'data', 'test'),
    os.path.join(BASE_DIR, 'test'),
    BASE_DIR,
    os.path.join(BASE_DIR, 'data'),
    '.'
])
if TEST_IMG_DIR is None:
    TEST_IMG_DIR = os.path.join(BASE_DIR, 'data', 'test')

# Verify image loading
test_im_check = cv2.imread(os.path.join(TEST_IMG_DIR, sample_test_img))
if test_im_check is not None:
    print(f"[OK] Verified TEST_IMG_DIR: {TEST_IMG_DIR} (sample resolution: {test_im_check.shape[1]}x{test_im_check.shape[0]})")
else:
    raise FileNotFoundError(
        f"Could not read the sample test image: {os.path.join(TEST_IMG_DIR, sample_test_img)}. "
        f"Resolved TEST_IMG_DIR={TEST_IMG_DIR}; check that the competition input is attached."
    )

TRAIN_IMG_DIR = None
if train_df is not None:
    sample_train_img = train_df.iloc[0]['image_1']
    TRAIN_IMG_DIR = resolve_image_root(sample_train_img, [
        os.path.join(BASE_DIR, 'data', 'train'),
        os.path.join(BASE_DIR, 'train'),
        BASE_DIR,
        os.path.join(BASE_DIR, 'data'),
        '.'
    ])
    if TRAIN_IMG_DIR is None:
        TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'data', 'train')
    train_im_check = cv2.imread(os.path.join(TRAIN_IMG_DIR, sample_train_img))
    if train_im_check is not None:
        print(f"[OK] Verified TRAIN_IMG_DIR: {TRAIN_IMG_DIR} (sample resolution: {train_im_check.shape[1]}x{train_im_check.shape[0]})")

# --- Step 4: Preflight every test image before expensive inference ---
def image_path(root, relative_path):
    return os.path.join(root, os.path.normpath(str(relative_path)))

missing_test_images = []
for _, row in test_df.iterrows():
    for column in ('image_1', 'image_2'):
        path = image_path(TEST_IMG_DIR, row[column])
        if not os.path.isfile(path):
            missing_test_images.append(path)

if missing_test_images:
    preview = '\n'.join(missing_test_images[:10])
    raise FileNotFoundError(
        f"{len(missing_test_images)} test image files are missing. First paths:\n{preview}"
    )
print(f"[OK] Preflight verified {len(test_df) * 2} test image paths.")

# --- Step 5: Define submission output path ---
if IS_KAGGLE:
    OUTPUT_CSV = '/kaggle/working/submission.csv'
else:
    OUTPUT_CSV = 'submission.csv'

print(f"\nPipeline Configuration Summary:")
print(f"  BASE_DIR      : {BASE_DIR}")
print(f"  TEST_CSV      : {TEST_CSV}")
print(f"  TEST_IMG_DIR  : {TEST_IMG_DIR}")
print(f"  TRAIN_CSV     : {TRAIN_CSV}")
print(f"  TRAIN_IMG_DIR : {TRAIN_IMG_DIR}")
print(f"  OUTPUT_CSV    : {OUTPUT_CSV}")

---
## 2. Evaluation Metric

Homographies are defined up to projective scale ($H \sim \lambda H$), so an element-wise matrix difference is geometrically meaningless.  
The competition evaluates **geometric reprojection error**:

1. Take 5 canonical normalized coordinates in Image 1:
   $$\{(0,0), (1,0), (1,1), (0,1), (0.5, 0.5)\}$$
2. Denormalize to pixel coordinates using Image 1's $(W_1, H_1)$.
3. Project into Image 2 using predicted homography $\hat{H}$ and ground-truth $H^*$.
4. Re-normalize to Image 2's $[0, 1] \times [0, 1]$ coordinate space by dividing by $(W_2, H_2)$.
5. Compute the mean Euclidean distance over the 5 points.

$$\text{Kaggle Leaderboard Score} = 100 \times \max\left(0, 1 - \frac{\text{Mean Error}}{0.2}\right)$$

In [ ]:
# 5 canonical normalized evaluation points
POINTS_NORM = np.array([
    [0.0, 0.0],
    [1.0, 0.0],
    [1.0, 1.0],
    [0.0, 1.0],
    [0.5, 0.5]
], dtype=np.float64)


def warp_points(H: np.ndarray, pts: np.ndarray) -> np.ndarray:
    """Warp 2D points using a 3x3 homography matrix H."""
    pts_homo = np.hstack([pts, np.ones((len(pts), 1), dtype=np.float64)])
    warped = (H @ pts_homo.T).T
    w = warped[:, 2:3]
    w = np.where(np.abs(w) < 1e-8, 1e-8, w)
    return warped[:, :2] / w


def compute_reprojection_error(pred_h: np.ndarray, gt_h: np.ndarray,
                               w1: int, h1: int, w2: int, h2: int) -> float:
    """Compute mean reprojection error across 5 normalized points."""
    pts_px_1 = POINTS_NORM * np.array([w1, h1], dtype=np.float64)
    pred_px_2 = warp_points(pred_h, pts_px_1)
    gt_px_2   = warp_points(gt_h, pts_px_1)
    pred_norm_2 = pred_px_2 / np.array([w2, h2], dtype=np.float64)
    gt_norm_2   = gt_px_2   / np.array([w2, h2], dtype=np.float64)
    return float(np.linalg.norm(pred_norm_2 - gt_norm_2, axis=1).mean())


def lb_score_from_error(mean_err: float) -> float:
    """Convert reprojection error into Kaggle 0-100 Leaderboard Score."""
    return float(100.0 * max(0.0, 1.0 - mean_err / 0.2))


def parse_h(row) -> np.ndarray:
    """Parse 3x3 homography matrix from CSV row."""
    return np.array([
        [float(row['h11']), float(row['h12']), float(row['h13'])],
        [float(row['h21']), float(row['h22']), float(row['h23'])],
        [float(row['h31']), float(row['h32']), 1.0]
    ], dtype=np.float64)

---
## 3. Exploratory Data Analysis & Identity Prior

A crucial insight from the dataset: approximately **50% of the training sequences** represent pure **illumination changes** (the camera remained stationary while lighting conditions shifted). For these pairs, the exact ground-truth is the identity matrix $H = I_{3 \times 3}$.

In [ ]:
if train_df is not None and not IS_KAGGLE:
    # Optional diagnostics are skipped on Kaggle to keep the submission run focused.
    # Identify pure identity pairs
    is_identity = (
        (train_df['h11'] == 1.0) & (train_df['h12'] == 0.0) & (train_df['h13'] == 0.0) &
        (train_df['h21'] == 0.0) & (train_df['h22'] == 1.0) & (train_df['h23'] == 0.0) &
        (train_df['h31'] == 0.0) & (train_df['h32'] == 0.0)
    )
    n_id = int(is_identity.sum())
    print(f"Pure Identity pairs in Train: {n_id} / {len(train_df)} ({n_id / len(train_df) * 100:.1f}%)")

    # Group by scene
    train_df['scene'] = train_df['pair_id'].str.rsplit('_', n=2).str[0] + '_' + train_df['pair_id'].str.rsplit('_', n=2).str[1]
    scene_records = {}
    for scene, g in train_df.groupby('scene'):
        h_vals = g[['h11','h12','h13','h21','h22','h23','h31','h32']].values
        is_illum = np.all(np.isclose(h_vals, [1,0,0,0,1,0,0,0], atol=1e-5))
        scene_records[scene] = 'illumination' if is_illum else 'viewpoint'

    scene_types = pd.Series(scene_records)
    print(f"\nScene Type Distribution:")
    print(scene_types.value_counts())
else:
    print("[INFO] train_df not available — skipping EDA identity analysis.")

In [ ]:
if train_df is not None and TRAIN_IMG_DIR is not None and not IS_KAGGLE:
    errors_identity = []
    for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Identity Baseline"):
        p1 = os.path.join(TRAIN_IMG_DIR, row['image_1'])
        p2 = os.path.join(TRAIN_IMG_DIR, row['image_2'])
        im1 = cv2.imread(p1)
        im2 = cv2.imread(p2)
        if im1 is None or im2 is None:
            continue
        h1, w1 = im1.shape[:2]
        h2, w2 = im2.shape[:2]
        H_gt = parse_h(row)
        errors_identity.append(compute_reprojection_error(np.eye(3), H_gt, w1, h1, w2, h2))

    mean_err_id = float(np.mean(errors_identity)) if errors_identity else 0.0
    print(f"\nIdentity Baseline Benchmark on Training Set:")
    print(f"  Mean Reprojection Error: {mean_err_id:.5f}")
    print(f"  Leaderboard Match Score: {lb_score_from_error(mean_err_id):.2f} / 100")
else:
    mean_err_id = 0.0
    print("[INFO] train_df not available — skipping identity baseline.")

In [ ]:
if train_df is not None and TRAIN_IMG_DIR is not None and not IS_KAGGLE and 'errors_identity' in locals() and errors_identity:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Error distribution of identity baseline
    axes[0].hist(errors_identity, bins=50, color='#4c72b0', edgecolor='white', alpha=0.85)
    axes[0].axvline(mean_err_id, color='red', linestyle='--', linewidth=2, label=f'Mean = {mean_err_id:.4f}')
    axes[0].set_xlabel('Reprojection Error', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    axes[0].set_title('Identity Baseline — Error Distribution', fontsize=13)
    axes[0].legend(fontsize=11)

    # Unique image dimensions
    dims = []
    for _, row in train_df.drop_duplicates('image_1').iterrows():
        im = cv2.imread(os.path.join(TRAIN_IMG_DIR, row['image_1']))
        if im is not None:
            dims.append((im.shape[1], im.shape[0]))
    dims = np.array(dims)
    if len(dims) > 0:
        axes[1].scatter(dims[:, 0], dims[:, 1], c='#55a868', s=60, alpha=0.7, edgecolors='white')
        axes[1].set_xlabel('Width (px)', fontsize=12)
        axes[1].set_ylabel('Height (px)', fontsize=12)
        axes[1].set_title('Image Dimensions across Training Scenes', fontsize=13)

    plt.tight_layout()
    plt.show()
else:
    print("[INFO] Skipping EDA plots.")

---
## 4. Multi-Stage Feature Matching & Geometric Guardrails

### Stage Architecture
| Stage | Detector | Pre-processing | Descriptor | Lowe Ratio | Estimator | Threshold | Fallback Condition |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **1** | SIFT (5K) | Grayscale | Standard SIFT | 0.75 | USAC_MAGSAC | 3.0 px | inliers < 15 or invalid |
| **2** | SIFT (8K) | CLAHE (clip 2.5) | **RootSIFT** | 0.75 | USAC_MAGSAC | 3.0 px | inliers < 10 or invalid |
| **3** | SIFT (8K) | CLAHE (clip 2.5) | Standard SIFT | 0.80 | USAC_MAGSAC | 5.0 px | inliers < 10 or invalid |
| **4** | SIFT (10K) | CLAHE (clip 3.5) | Standard SIFT | 0.85 | USAC_MAGSAC | 5.0 px | inliers < 10 or invalid |
| **Fallback** | — | — | — | — | — | — | Defaults to $I_{3 \times 3}$ |

### Geometric Guardrails
Every candidate homography $H$ must pass 5 physical validity checks before acceptance:
1. **Finite values**: No NaNs or Infs.
2. **Determinant range**: $0.005 < |\det(H)| < 50.0$ (disallows extreme perspective folding/flips).
3. **Positive Projective Depth (Cheirality)**: $w_i' > 0$ for all 4 warped image corners (points remain in front of the camera).
4. **Convex Quadrilateral**: Cross-products along warped boundary vertices maintain identical signs (no self-intersecting or pinched quads).
5. **Bounding Box Bounds**: Warped bounding box must be at least $10 \times 10$ px and no larger than $20 \times$ the target image size.

In [ ]:
# ======================================================================
# GEOMETRIC VALIDATION UTILITIES
# ======================================================================

def is_convex_quad(pts: np.ndarray) -> bool:
    """Check if 4 points form a strictly convex polygon with no self-intersections."""
    cross_products = []
    for i in range(4):
        p1 = pts[i]
        p2 = pts[(i + 1) % 4]
        p3 = pts[(i + 2) % 4]
        v1 = p2 - p1
        v2 = p3 - p2
        cp = v1[0] * v2[1] - v1[1] * v2[0]
        cross_products.append(cp)
    return all(cp > 0 for cp in cross_products) or all(cp < 0 for cp in cross_products)


def is_valid_homography(H: np.ndarray, w1: int, h1: int, w2: int, h2: int) -> bool:
    """
    Validate physical and geometric feasibility of the predicted homography:
    - Finite numerical values
    - Determinant within realistic perspective transformation bounds
    - Positive projective depth (w > 0) for all 4 warped corners
    - Warped boundary forms a valid convex quadrilateral
    - Warped bounding box within reasonable coordinate bounds
    """
    if H is None or not np.isfinite(H).all():
        return False

    det = np.linalg.det(H)
    if not (0.005 < abs(det) < 50.0):
        return False

    corners_1 = np.array([
        [0.0, 0.0, 1.0],
        [float(w1), 0.0, 1.0],
        [float(w1), float(h1), 1.0],
        [0.0, float(h1), 1.0]
    ], dtype=np.float64)

    warped_homo = (H @ corners_1.T).T
    if (warped_homo[:, 2] <= 1e-4).any():
        return False

    warped_pts = warped_homo[:, :2] / warped_homo[:, 2:3]
    if not is_convex_quad(warped_pts):
        return False

    x_min, y_min = warped_pts.min(axis=0)
    x_max, y_max = warped_pts.max(axis=0)
    box_w = x_max - x_min
    box_h = y_max - y_min
    if box_w < 10 or box_h < 10:
        return False
    if box_w > 20 * w2 or box_h > 20 * h2:
        return False

    return True

In [ ]:
# ======================================================================
# FEATURE EXTRACTION & DESCRIPTOR NORMALIZATION
# ======================================================================

def rootsift(des: np.ndarray) -> np.ndarray:
    """
    RootSIFT: Hellinger kernel descriptor transformation.
    L1-normalize -> element-wise sqrt -> L2-normalize.
    Significantly suppresses visual burstiness and false matches.
    """
    if des is None or len(des) == 0:
        return des
    des_norm = des / (np.linalg.norm(des, axis=1, ord=1, keepdims=True) + 1e-7)
    des_sqrt = np.sqrt(des_norm)
    des_l2   = des_sqrt / (np.linalg.norm(des_sqrt, axis=1, ord=2, keepdims=True) + 1e-7)
    return des_l2.astype(np.float32)


# Pre-allocate feature detectors, matchers, and contrast filters
sift_standard   = cv2.SIFT_create(nfeatures=5000,  contrastThreshold=0.015, edgeThreshold=10)
sift_sensitive  = cv2.SIFT_create(nfeatures=8000,  contrastThreshold=0.008, edgeThreshold=10)
sift_aggressive = cv2.SIFT_create(nfeatures=10000, contrastThreshold=0.005, edgeThreshold=10)

bf_matcher   = cv2.BFMatcher(cv2.NORM_L2)
clahe_filter = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
clahe_strong = cv2.createCLAHE(clipLimit=3.5, tileGridSize=(8, 8))

# Robust estimator configuration
RANSAC_METHOD = getattr(cv2, 'USAC_MAGSAC', cv2.RANSAC)
print(f"Robust Estimator: {'cv2.USAC_MAGSAC' if RANSAC_METHOD == getattr(cv2, 'USAC_MAGSAC', None) else 'cv2.RANSAC'}")

In [ ]:
# ======================================================================
# MATCHING & ROBUST HOMOGRAPHY ESTIMATION
# ======================================================================

def match_and_estimate(kp1, des1, kp2, des2,
                       w1: int, h1: int, w2: int, h2: int,
                       ratio: float = 0.75, ransac_thresh: float = 3.0):
    """
    Lowe ratio test -> USAC_MAGSAC robust estimation -> geometric validation.
    Returns (H_normalized, inlier_count) or (None, 0) on failure.
    """
    if des1 is None or des2 is None or len(des1) < 4 or len(des2) < 4:
        return None, 0

    matches = bf_matcher.knnMatch(des1, des2, k=2)
    good = [m for m, n in matches if len((m, n)) == 2 and m.distance < ratio * n.distance]
    if len(good) < 4:
        return None, 0

    src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

    H, mask = cv2.findHomography(src_pts, dst_pts, RANSAC_METHOD, ransac_thresh)
    inliers = int(mask.sum()) if mask is not None else 0

    if H is not None and np.isfinite(H).all() and abs(H[2, 2]) > 1e-8:
        H_norm = H / H[2, 2]
        if is_valid_homography(H_norm, w1, h1, w2, h2):
            return H_norm, inliers

    return None, 0

In [ ]:
# ======================================================================
# MAIN PREDICTION PIPELINE
# ======================================================================

def predict(image_1_path: str, image_2_path: str) -> np.ndarray:
    """
    Predict the 3x3 homography matrix mapping Image 1 -> Image 2.
    Returns np.ndarray of shape (3, 3) with H[2, 2] == 1.0.
    """
    img1 = cv2.imread(image_1_path, cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(image_2_path, cv2.IMREAD_GRAYSCALE)

    if img1 is None or img2 is None:
        missing = []
        if img1 is None: missing.append(image_1_path)
        if img2 is None: missing.append(image_2_path)
        if not hasattr(predict, '_warned_paths'):
            predict._warned_paths = set()
        for m in missing:
            if m not in predict._warned_paths:
                print(f"[WARNING] Could not read image: '{m}' -> falling back to identity matrix.")
                predict._warned_paths.add(m)
        return np.eye(3, dtype=np.float64)

    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    # ---- Stage 1: Standard SIFT (ratio 0.75, thresh 3.0) ----
    kp1, des1 = sift_standard.detectAndCompute(img1, None)
    kp2, des2 = sift_standard.detectAndCompute(img2, None)
    H_pred, inliers = match_and_estimate(
        kp1, des1, kp2, des2, w1, h1, w2, h2, ratio=0.75, ransac_thresh=3.0
    )

    # ---- Stage 2: CLAHE + RootSIFT (Hellinger kernel for lighting variations) ----
    if H_pred is None or inliers < 15:
        img1_c = clahe_filter.apply(img1)
        img2_c = clahe_filter.apply(img2)
        kp1_c, des1_c = sift_sensitive.detectAndCompute(img1_c, None)
        kp2_c, des2_c = sift_sensitive.detectAndCompute(img2_c, None)
        H_c, inliers_c = match_and_estimate(
            kp1_c, rootsift(des1_c), kp2_c, rootsift(des2_c),
            w1, h1, w2, h2, ratio=0.75, ransac_thresh=3.0
        )
        if H_c is not None and inliers_c > max(inliers, 0):
            H_pred, inliers = H_c, inliers_c

    # ---- Stage 3: CLAHE + standard SIFT (relaxed ratio 0.80, thresh 5.0) ----
    if H_pred is None or inliers < 10:
        img1_c = clahe_filter.apply(img1)
        img2_c = clahe_filter.apply(img2)
        kp1_c, des1_c = sift_sensitive.detectAndCompute(img1_c, None)
        kp2_c, des2_c = sift_sensitive.detectAndCompute(img2_c, None)
        H_c, inliers_c = match_and_estimate(
            kp1_c, des1_c, kp2_c, des2_c,
            w1, h1, w2, h2, ratio=0.80, ransac_thresh=5.0
        )
        if H_c is not None and inliers_c > max(inliers, 0):
            H_pred, inliers = H_c, inliers_c

    # ---- Stage 4: High-contrast CLAHE + ultra-sensitive SIFT (ratio 0.85) ----
    if H_pred is None or inliers < 10:
        img1_c = clahe_strong.apply(img1)
        img2_c = clahe_strong.apply(img2)
        kp1_c, des1_c = sift_aggressive.detectAndCompute(img1_c, None)
        kp2_c, des2_c = sift_aggressive.detectAndCompute(img2_c, None)
        H_c, inliers_c = match_and_estimate(
            kp1_c, des1_c, kp2_c, des2_c,
            w1, h1, w2, h2, ratio=0.85, ransac_thresh=5.0
        )
        if H_c is not None and inliers_c > max(inliers, 0):
            H_pred, inliers = H_c, inliers_c

    # ---- Stage 5: Identity prior fallback ----
    if H_pred is None or inliers < 10:
        H_pred = np.eye(3, dtype=np.float64)

    # Standardize H[2, 2] == 1.0
    H_pred = H_pred / H_pred[2, 2]
    return H_pred.astype(np.float64)

---
## 5. Local Validation Benchmark

Running the complete pipeline on all 140 training pairs to evaluate against the ground-truth homographies.

In [ ]:
if train_df is not None and TRAIN_IMG_DIR is not None and not IS_KAGGLE:
    train_errors = []
    train_predictions = []

    for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Evaluating Train Benchmark"):
        p1 = os.path.join(TRAIN_IMG_DIR, row['image_1'])
        p2 = os.path.join(TRAIN_IMG_DIR, row['image_2'])
        im1 = cv2.imread(p1)
        im2 = cv2.imread(p2)
        if im1 is None or im2 is None:
            continue
        h1, w1 = im1.shape[:2]
        h2, w2 = im2.shape[:2]

        H_gt   = parse_h(row)
        H_pred = predict(p1, p2)
        err    = compute_reprojection_error(H_pred, H_gt, w1, h1, w2, h2)

        train_errors.append(err)
        train_predictions.append({
            'pair_id': row['pair_id'],
            'h11': H_pred[0, 0], 'h12': H_pred[0, 1], 'h13': H_pred[0, 2],
            'h21': H_pred[1, 0], 'h22': H_pred[1, 1], 'h23': H_pred[1, 2],
            'h31': H_pred[2, 0], 'h32': H_pred[2, 1],
            'error': err
        })

    mean_train_err = float(np.mean(train_errors)) if train_errors else 0.0
    train_score = lb_score_from_error(mean_train_err)

    print(f"\n{'=' * 65}")
    print(f"  LOCAL VALIDATION BENCHMARK RESULTS")
    print(f"{'=' * 65}")
    print(f"  Mean Reprojection Error : {mean_train_err:.5f}  (lower is better)")
    print(f"  Kaggle Leaderboard Score: {train_score:.2f} / 100  (higher is better)")
    if 'mean_err_id' in locals() and mean_err_id > 0:
        print(f"  Improvement over Identity: +{train_score - lb_score_from_error(mean_err_id):.2f} points")
    print(f"{'=' * 65}")
else:
    print("[INFO] train_df not available — skipping local validation benchmark.")

In [ ]:
if train_df is not None and not IS_KAGGLE and 'train_predictions' in locals() and train_predictions:
    pred_analysis = pd.DataFrame(train_predictions)
    pred_analysis['scene'] = pred_analysis['pair_id'].str.rsplit('_', n=2).str[0] + '_' + pred_analysis['pair_id'].str.rsplit('_', n=2).str[1]

    print("\nTop 10 Worst Pairs (highest reprojection error):")
    print(pred_analysis.nlargest(10, 'error')[['pair_id', 'error']].to_string(index=False))

    fig, ax = plt.subplots(figsize=(12, 5))
    if 'errors_identity' in locals() and errors_identity:
        ax.hist(errors_identity, bins=50, alpha=0.45, color='#dd8452', label=f'Identity (score={lb_score_from_error(mean_err_id):.1f})', edgecolor='white')
        ax.axvline(mean_err_id, color='orange', linestyle='--', linewidth=2)
    ax.hist(train_errors, bins=50, alpha=0.6, color='#4c72b0', label=f'Our Pipeline (score={train_score:.1f})', edgecolor='white')
    ax.axvline(mean_train_err, color='blue', linestyle='--', linewidth=2)
    ax.set_xlabel('Reprojection Error', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Error Distribution: Multi-Stage Pipeline vs. Identity Prior', fontsize=13)
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print("[INFO] Skipping validation distribution plot.")

---
## 6. Qualitative Visualizations

Inspecting SIFT keypoint matches and perspective alignment via warped overlay on a viewpoint-shifted pair.

In [ ]:
if train_df is not None and TRAIN_IMG_DIR is not None and not IS_KAGGLE and 'is_identity' in locals() and (~is_identity).any():
    viewpoint_pairs = train_df[~is_identity]
    sample_row = viewpoint_pairs.iloc[0]
    p1 = os.path.join(TRAIN_IMG_DIR, sample_row['image_1'])
    p2 = os.path.join(TRAIN_IMG_DIR, sample_row['image_2'])
    has_gt = True
    H_gt_vis = parse_h(sample_row)
else:
    sample_row = test_df.iloc[0]
    p1 = os.path.join(TEST_IMG_DIR, sample_row['image_1'])
    p2 = os.path.join(TEST_IMG_DIR, sample_row['image_2'])
    has_gt = False

im1_bgr = cv2.imread(p1)
im2_bgr = cv2.imread(p2)

if im1_bgr is not None and im2_bgr is not None:
    h1_vis, w1_vis = im1_bgr.shape[:2]
    h2_vis, w2_vis = im2_bgr.shape[:2]
    H_vis = predict(p1, p2)
    print(f"Sample Pair: {sample_row['pair_id']}")
    if has_gt:
        err_vis = compute_reprojection_error(H_vis, H_gt_vis, w1_vis, h1_vis, w2_vis, h2_vis)
        print(f"Reprojection Error: {err_vis:.6f}")
        print(f"\nPredicted H:\n{np.array2string(H_vis, precision=4, suppress_small=True)}")
        print(f"\nGround Truth H:\n{np.array2string(H_gt_vis, precision=4, suppress_small=True)}")
    else:
        print(f"\nPredicted H:\n{np.array2string(H_vis, precision=4, suppress_small=True)}")
else:
    print(f"[WARNING] Could not load sample images for visualization.")

In [ ]:
if not IS_KAGGLE and 'im1_bgr' in locals() and im1_bgr is not None and im2_bgr is not None:
    warped_im1 = cv2.warpPerspective(im1_bgr, H_vis, (w2_vis, h2_vis))
    overlay = cv2.addWeighted(im2_bgr, 0.5, warped_im1, 0.5, 0)

    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    axes[0].imshow(cv2.cvtColor(im1_bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Image 1 (Reference)', fontsize=13)
    axes[0].axis('off')

    axes[1].imshow(cv2.cvtColor(im2_bgr, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Image 2 (Target)', fontsize=13)
    axes[1].axis('off')

    axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    suffix = f" (err={err_vis:.5f})" if 'err_vis' in locals() else ""
    axes[2].set_title(f'Aligned Overlay{suffix}', fontsize=13)
    axes[2].axis('off')

    plt.suptitle(f'Pair: {sample_row["pair_id"]}', fontsize=15, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
if not IS_KAGGLE and 'im1_bgr' in locals() and im1_bgr is not None and im2_bgr is not None:
    img1_gray = cv2.cvtColor(im1_bgr, cv2.COLOR_BGR2GRAY)
    img2_gray = cv2.cvtColor(im2_bgr, cv2.COLOR_BGR2GRAY)

    kp1_v, des1_v = sift_standard.detectAndCompute(img1_gray, None)
    kp2_v, des2_v = sift_standard.detectAndCompute(img2_gray, None)

    if des1_v is not None and des2_v is not None and len(des1_v) >= 2 and len(des2_v) >= 2:
        matches_v = bf_matcher.knnMatch(des1_v, des2_v, k=2)
        good_v = [m for m, n in matches_v if len((m, n)) == 2 and m.distance < 0.75 * n.distance]

        match_img = cv2.drawMatches(
            img1_gray, kp1_v, img2_gray, kp2_v,
            good_v[:50], None,
            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
            matchColor=(0, 255, 0)
        )

        fig, ax = plt.subplots(figsize=(18, 7))
        ax.imshow(match_img)
        ax.set_title(f'SIFT Inlier Keypoint Matches ({len(good_v)} total, showing top 50)', fontsize=14)
        ax.axis('off')
        plt.tight_layout()
        plt.show()

---
## 7. Test Set Inference & Generating `submission.csv`

Runs predictions across all pairs in `test.csv` and outputs the competition-compliant CSV file with schema:  
`pair_id,h11,h12,h13,h21,h22,h23,h31,h32`

In [ ]:
submission_rows = []
start_time = time.time()

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test Inference"):
    pair_id = row['pair_id']
    p1 = os.path.join(TEST_IMG_DIR, row['image_1'])
    p2 = os.path.join(TEST_IMG_DIR, row['image_2'])

    H = predict(p1, p2)

    submission_rows.append({
        'pair_id': pair_id,
        'h11': float(H[0, 0]),
        'h12': float(H[0, 1]),
        'h13': float(H[0, 2]),
        'h21': float(H[1, 0]),
        'h22': float(H[1, 1]),
        'h23': float(H[1, 2]),
        'h31': float(H[2, 0]),
        'h32': float(H[2, 1])
    })

elapsed = time.time() - start_time

sub_df = pd.DataFrame(submission_rows)
cols = ['pair_id', 'h11', 'h12', 'h13', 'h21', 'h22', 'h23', 'h31', 'h32']
sub_df = sub_df[cols]

# Write to destination
sub_df.to_csv(OUTPUT_CSV, index=False)

print(f"\n{'=' * 70}")
print(f"  SAVED PREDICTIONS TO: {OUTPUT_CSV}")
print(f"  Total inference time : {elapsed:.2f}s  ({elapsed / len(test_df):.3f}s / pair)")
print(f"  Output shape         : {sub_df.shape}  ({len(sub_df)} rows, {len(cols)} columns)")
print(f"  File size            : {os.path.getsize(OUTPUT_CSV)} bytes")
print(f"{'=' * 70}")
sub_df.head(10)

---
## 8. Automated Submission Verification & Kaggle Upload Guide

Rigorous sanity checks on the generated submission:
1. Column header names and ordering
2. Total row count matching `test.csv`
3. Exact 1-to-1 `pair_id` alignment
4. Absence of nulls, NaNs, or Infs
5. Invertibility check (determinant non-zero for all homographies)

In [ ]:
# ======================================================================
# COMPLETE SUBMISSION VALIDATION & SUBMIT INSTRUCTIONS
# ======================================================================

EXPECTED_COLS = ['pair_id', 'h11', 'h12', 'h13', 'h21', 'h22', 'h23', 'h31', 'h32']

sub_check  = pd.read_csv(OUTPUT_CSV)
test_check = pd.read_csv(TEST_CSV)

all_ok = True

# 1. Column check
if list(sub_check.columns) == EXPECTED_COLS:
    print("  [OK] Column headers match expected format exactly.")
else:
    print(f"  [ERROR] Columns mismatch! Expected {EXPECTED_COLS}, got {list(sub_check.columns)}")
    all_ok = False

# 2. Row count check
if len(sub_check) == len(test_check):
    print(f"  [OK] Row count matches test.csv ({len(sub_check)} rows).")
else:
    print(f"  [ERROR] Row count mismatch! Expected {len(test_check)}, got {len(sub_check)}.")
    all_ok = False

# 3. pair_id alignment
if (sub_check['pair_id'].values == test_check['pair_id'].values).all():
    print("  [OK] pair_id values and sequential order match test.csv perfectly.")
else:
    print("  [ERROR] pair_id values or order do not match test.csv!")
    all_ok = False

# 4. NaN / Inf checks
has_null = False
has_inf  = False
for col in EXPECTED_COLS[1:]:
    if sub_check[col].isnull().any():
        print(f"  [ERROR] Column '{col}' contains NaN values!")
        has_null = True
    if np.isinf(sub_check[col]).any():
        print(f"  [ERROR] Column '{col}' contains Inf values!")
        has_inf = True

if not has_null and not has_inf:
    print("  [OK] No nulls, NaNs, or Infinite values found.")
else:
    all_ok = False

# 5. Non-degeneracy & determinant check
degenerate = 0
for _, r in sub_check.iterrows():
    H_check = np.array([
        [float(r['h11']), float(r['h12']), float(r['h13'])],
        [float(r['h21']), float(r['h22']), float(r['h23'])],
        [float(r['h31']), float(r['h32']), 1.0]
    ])
    det = np.linalg.det(H_check)
    if abs(det) < 1e-8:
        degenerate += 1
        print(f"  [ERROR] Pair {r['pair_id']} has near-zero determinant ({det:.2e})!")

if degenerate == 0:
    print("  [OK] All homography matrices are non-singular and strictly invertible.")
else:
    print(f"  [ERROR] Found {degenerate} singular/degenerate matrices.")
    all_ok = False

if all_ok:
    print("\n" + "=" * 76)
    print("  ✅ SUCCESS: submission.csv is 100% VALID and ready for Kaggle submission!")
    print("=" * 76)
    if IS_KAGGLE:
        print("\n👉 HOW TO SUBMIT ON KAGGLE:")
        print("   1. Open the right-hand sidebar of this Kaggle notebook.")
        print("   2. Look under the 'Output' section -> '/kaggle/working/'.")
        print("   3. Locate 'submission.csv'.")
        print("   4. Click the 'Submit to Competition' button (or click the 3 dots '...' -> 'Submit').")
    else:
        print(f"\nSubmission file ready locally at: {os.path.abspath(OUTPUT_CSV)}")
else:
    print("\n❌ FAILED: Please fix the errors listed above before submitting.")

# Interactive download link in Jupyter / Kaggle
try:
    from IPython.display import FileLink, display
    display(FileLink(OUTPUT_CSV))
except Exception:
    pass